# Full, Partial, Linear Probe, and Adapter Tuning

LoRA and QLoRA are popular, but they sit inside a wider family of adaptation methods. Before choosing LoRA by default, you should understand the baseline options: full fine-tuning, feature extraction, linear probing, partial fine-tuning, and classic adapters.

This notebook compares those approaches with small PyTorch modules so the trade-offs are visible.

## What You Will Learn

By the end, you should understand:

- what full fine-tuning updates,
- what feature extraction means,
- how a linear probe works,
- how partial fine-tuning freezes and unfreezes layers,
- how bottleneck adapters work,
- how trainable parameter count changes by method,
- and when each method is a reasonable baseline.

## 1. Adaptation Strategy Map

| Method | Trainable Parameters | Typical Cost | Use When |
|---|---|---|---|
| Full fine-tuning | All weights | Highest | You have compute and need maximum adaptation |
| Feature extraction | New head only | Low | Base representation is already useful |
| Linear probe | Linear classifier/regressor only | Very low | You want a fast representation test |
| Partial fine-tuning | Selected layers | Medium | Top layers need task adaptation |
| Bottleneck adapters | Small inserted modules | Low-medium | You want modular adapters without LoRA |
| LoRA | Low-rank updates | Low | Strong default for LLM adaptation |

A good fine-tuning workflow often starts with the cheapest baseline that could work, then scales only if needed.

In [ ]:
import copy
from dataclasses import dataclass

import matplotlib.pyplot as plt
import pandas as pd
import torch
from torch import nn
from torch.nn import functional as F

torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

## 2. A Tiny Base Model

We will use a tiny encoder-style network as a stand-in for a pretrained model. In a real LLM, the blocks would be transformer blocks and the head would be a language modeling or classification head.

In [ ]:
class TinyBlock(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.LayerNorm(dim),
            nn.Linear(dim, dim * 2),
            nn.GELU(),
            nn.Linear(dim * 2, dim),
        )

    def forward(self, x):
        return x + self.net(x)

class TinyEncoder(nn.Module):
    def __init__(self, input_dim=16, hidden_dim=64, num_blocks=4, num_classes=2):
        super().__init__()
        self.input_proj = nn.Linear(input_dim, hidden_dim)
        self.blocks = nn.ModuleList([TinyBlock(hidden_dim) for _ in range(num_blocks)])
        self.head = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        x = self.input_proj(x)
        for block in self.blocks:
            x = block(x)
        return self.head(x)

base_model = TinyEncoder().to(device)
print(base_model)

In [ ]:
def count_parameters(model):
    total = sum(parameter.numel() for parameter in model.parameters())
    trainable = sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)
    return total, trainable, 100 * trainable / total

def freeze_all(model):
    for parameter in model.parameters():
        parameter.requires_grad = False

def unfreeze_module(module):
    for parameter in module.parameters():
        parameter.requires_grad = True

def parameter_report(name, model):
    total, trainable, percent = count_parameters(model)
    return {"method": name, "total_params": total, "trainable_params": trainable, "trainable_percent": percent}

parameter_report("base_all_trainable", base_model)

## 3. Full Fine-Tuning

Full fine-tuning updates every parameter. This gives maximum flexibility, but it also requires the most memory because gradients and optimizer states exist for all trainable weights. It also has a higher risk of overfitting or damaging general behavior.

In [ ]:
full_model = copy.deepcopy(base_model)
for parameter in full_model.parameters():
    parameter.requires_grad = True
parameter_report("full_finetuning", full_model)

## 4. Feature Extraction and Linear Probe

Feature extraction freezes the base model and trains only a new head. A linear probe is the simplest version: a linear classifier or regressor on top of frozen representations.

This is a great baseline because it answers: are the base model representations already useful for this task?

In [ ]:
linear_probe_model = copy.deepcopy(base_model)
freeze_all(linear_probe_model)
linear_probe_model.head = nn.Linear(64, 2).to(device)
unfreeze_module(linear_probe_model.head)
parameter_report("linear_probe", linear_probe_model)

## 5. Partial Fine-Tuning

Partial fine-tuning freezes most of the model and unfreezes selected layers, often the final blocks and head. It is more flexible than a linear probe and cheaper than full fine-tuning.

For LLMs, partial tuning can mean unfreezing the final transformer blocks, only layer norms, only embeddings, or only the language modeling head.

In [ ]:
partial_model = copy.deepcopy(base_model)
freeze_all(partial_model)
unfreeze_module(partial_model.blocks[-1])
unfreeze_module(partial_model.head)
parameter_report("partial_last_block_plus_head", partial_model)

## 6. Bottleneck Adapters

Classic adapters insert small trainable bottleneck modules into frozen layers. A simple adapter projects down to a small hidden dimension, applies a nonlinearity, projects back up, and adds the result to the original hidden state.

```text
adapter(x) = x + W_up activation(W_down x)
```

This is different from LoRA: adapters add a small module in the forward path, while LoRA modifies selected weight matrices with low-rank updates.

In [ ]:
class BottleneckAdapter(nn.Module):
    def __init__(self, dim, bottleneck_dim=8):
        super().__init__()
        self.down = nn.Linear(dim, bottleneck_dim)
        self.activation = nn.GELU()
        self.up = nn.Linear(bottleneck_dim, dim)
        nn.init.zeros_(self.up.weight)
        nn.init.zeros_(self.up.bias)

    def forward(self, x):
        return x + self.up(self.activation(self.down(x)))

class TinyBlockWithAdapter(nn.Module):
    def __init__(self, base_block, dim=64, bottleneck_dim=8):
        super().__init__()
        self.base_block = base_block
        self.adapter = BottleneckAdapter(dim, bottleneck_dim)

    def forward(self, x):
        x = self.base_block(x)
        return self.adapter(x)

adapter_model = copy.deepcopy(base_model)
freeze_all(adapter_model)
for index, block in enumerate(adapter_model.blocks):
    adapter_model.blocks[index] = TinyBlockWithAdapter(block).to(device)
for module in adapter_model.modules():
    if isinstance(module, BottleneckAdapter):
        unfreeze_module(module)
unfreeze_module(adapter_model.head)
parameter_report("bottleneck_adapters_plus_head", adapter_model)

## 7. Compare Trainable Parameter Counts

Trainable parameter count is not the only cost, but it is the first thing to inspect. Optimizer states often use more memory than the weights themselves.

In [ ]:
reports = [
    parameter_report("full_finetuning", full_model),
    parameter_report("linear_probe", linear_probe_model),
    parameter_report("partial_last_block_plus_head", partial_model),
    parameter_report("bottleneck_adapters_plus_head", adapter_model),
]
report_df = pd.DataFrame(reports)
report_df

In [ ]:
ax = report_df.set_index("method")["trainable_percent"].plot(kind="bar", figsize=(9, 4))
ax.set_ylabel("trainable parameters (%)")
ax.set_title("Trainable parameter share by adaptation method")
ax.grid(axis="y", alpha=0.3)
plt.xticks(rotation=25, ha="right")
plt.tight_layout()

## 8. Tiny Training Comparison

We can train each method on a synthetic classification problem. This is not meant to prove which method is universally better. It shows how to run a controlled baseline comparison.

In [ ]:
num_samples = 512
features = torch.randn(num_samples, 16, device=device)
true_w = torch.randn(16, 2, device=device)
labels = (features @ true_w).argmax(dim=-1)

def train_small_model(model, steps=80, lr=2e-3):
    model = model.to(device)
    optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=lr)
    losses = []
    for _ in range(steps):
        logits = model(features)
        loss = F.cross_entropy(logits, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
    with torch.inference_mode():
        accuracy = (model(features).argmax(dim=-1) == labels).float().mean().item()
    return losses, accuracy

results = []
for name, model in [
    ("linear_probe", linear_probe_model),
    ("partial", partial_model),
    ("adapters", adapter_model),
    ("full", full_model),
]:
    losses, accuracy = train_small_model(copy.deepcopy(model))
    results.append({"method": name, "final_loss": losses[-1], "train_accuracy": accuracy})
pd.DataFrame(results)

## 9. Decision Guide

| Situation | Start With | Why |
|---|---|---|
| You only need a classifier on top of strong embeddings | Linear probe | Fast and cheap baseline |
| The model mostly knows the task but needs a small shift | Partial tuning or LoRA | Less risk than full tuning |
| You need modular domain adapters | Adapters or LoRA | Easy to save and swap |
| You have a large LLM and limited GPU memory | QLoRA | Best memory trade-off |
| You have lots of high-quality data and compute | Full fine-tuning | Maximum adaptation capacity |
| You are unsure whether data helps | Linear probe or small LoRA pilot | Cheap signal before scaling |

## 10. Practical Notes for LLMs

- Full LLM fine-tuning needs optimizer state for all weights, so memory grows quickly.
- Partial tuning can be unstable if only a few layers adapt to a very different task.
- Linear probes are more common for encoder embeddings than decoder-only chat models.
- LoRA is the usual default for decoder-only LLM SFT because it is strong and simple.
- Classic adapters are still conceptually useful and appear in many older PEFT systems.
- Always compare against a frozen or low-cost baseline before doing expensive training.

## Summary

Full fine-tuning, feature extraction, linear probes, partial tuning, adapters, and LoRA are all points on the same trade-off curve: adaptation capacity vs memory, cost, modularity, and risk. LoRA is a strong default for LLMs, but baselines matter.

Next, we move from supervised examples to preference data with DPO and reward modeling.